# 📚 Fiber Segmentation Tutorial with SAM2

This tutorial demonstrates the segmentation and structural analysis of fiber cross-section micrographs using SAM2 — the Segment Anything Model v2 developed by Meta (Facebook) Research.
SAM2 leverages Vision Transformers (ViT) to achieve highly flexible, prompt-based instance segmentation.

The workflow begins with the Automatic Mask Generator, which segments individual fiber cross sections in a single micrograph.
Subsequently, the notebook introduces a path extraction routine that subdivides large panoramic images into smaller, overlapping patches for detailed and systematic analysis.

This step-by-step guide provides a practical introduction to automated fiber segmentation and quantitative microstructural characterization.

📝 **Notation**. In the following, **image** refers to the original grayscale or RGB input, **mask** denotes the binary segmentation map highlighting the foreground region, and instance designates the **labeled map** in which each distinct **object** or individual **mask** is assigned a unique integer identifier.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/choROPeNt/segment-anything-2/blob/dev/notebooks/colab_analysis.ipynb)

## 🌍 Environment Set-up
If running locally using jupyter, first install `SAM 2` in your environment using the installation instructions in the repository.

If running from Google Colab, set `using_colab=True` below and run the cell. In Colab, be sure to select 'GPU' under 'Edit'->'Notebook Settings'->'Hardware accelerator'.
> Note that it's recommended to use **A100 or L4 GPUs when running in Colab** (T4 GPUs might also work, but could be slow and might run out of memory in some cases e.g. training).

In [ ]:
using_colab = True # set to False if you are not working on Google Colab

### 📦 Load Packages

If you are on Google Colab some packages have to be installed that might take up to 5 min. Grab a a Coffe ☕️

❗️**THE FOLOWING CELL HAS TO BE EXECUTED TO LOAD ALL NECESSARY PACKAGES**❗️



In [ ]:
%%capture
# --- Packages --
import os
import sys
import torch
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage.measure import regionprops_table, regionprops
import pandas as pd
from tqdm.auto import tqdm
from time import time

# ===========================================================
#                Google Colab Setup (optional)
# ===========================================================
if using_colab:
    # Install required packages
    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/choROPeNt/segment-anything-2.git@dev'

    # Download example images
    !mkdir -p ./images/
    !wget -P ./images/ https://raw.githubusercontent.com/choROPeNt/segment-anything-2/dev/images/image_cfrp.tiff
    !wget -P ./images/ https://raw.githubusercontent.com/choROPeNt/segment-anything-2/dev/images/image_gfrp.tiff
    !wget -P ./images/ https://raw.githubusercontent.com/choROPeNt/segment-anything-2/dev/images/panorama.tif

    # Download SAM2 model checkpoint
    !mkdir -p ./checkpoints/
    !wget -P ./checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt
else:
    import sam2


## 🏎️ Device Selection

PyTorch currently supports the following hardware acceleration backends and are recommended:

- **CUDA** — NVIDIA GPU acceleration (Linux and Windows)
- **MPS** — Metal Performance Shaders for Apple Silicon (macOS)
- **ROCm** — AMD GPU acceleration (Linux)

Other backends such as **XPU** (Intel) or **TPU** (Google Cloud) require separate extensions and are not part of the core PyTorch package.

| Backend    | Platform    |  PyTorch support   |  Notes   |
| --- | --- | --- | --- |
| `CUDA `   |  Linux / Windows   |  ✅ Fully supported   |  Standard GPU backend for NVIDIA cards.   |   
| `MPS`    |   macOS (Apple Silicon or AMD GPUs)  |  ✅ Supported (since 1.12)   |  Uses Apple Metal; some ops may still fall back to CPU. Some numerical differences to `CUDA`  |     
|  `ROCm`   |   Linux (AMD GPUs)  |  ✅ Supported (since 1.10, stable ≥ 1.12)   |  Official ROCm builds available from PyTorch.org.   |     
|   `XPU / SYCL`  |  Intel GPUs   |  🧪 Experimental   |  Via Intel Extension for PyTorch (not in core).   |    
|  `TPU`   |  Cloud (Google Colab `TPU` VMs)   |  ❌ Not in core PyTorch   |   Requires torch_xla separately.  |    





























In [ ]:
# -----------------------------------------------------------
# Select device (CUDA → MPS → CPU)
# -----------------------------------------------------------

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"  # fallback for unsupported ops
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using PyTorch Version {torch.__version__}")
print(f"Using device: {device}")

# -----------------------------------------------------------
# CUDA-specific optimizations
# -----------------------------------------------------------
if device.type == "cuda":
    # Enable bfloat16 autocast for entire notebook
    torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

    # Enable TensorFloat-32 (TF32) on Ampere or newer GPUs
    props = torch.cuda.get_device_properties(0)
    if props.major >= 8:  # Ampere or newer
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

# -----------------------------------------------------------
# MPS warning (Apple Silicon)
# -----------------------------------------------------------
elif device.type == "mps":
    print(
        "\n⚠️  Warning: MPS support in PyTorch is still experimental.\n"
        "    SAM2 is trained on CUDA and may produce slightly different\n"
        "    numerical results or slower performance on MPS.\n"
        "    More details: https://github.com/pytorch/pytorch/issues/84936\n"
    )

#  1️⃣ Part 1: Single Image Analysis

## 💿 Load Image

**Note:**  
SAM2 expects images in the format **`[W, H, 3]`**, where the last dimension corresponds to the **RGB color channels**.

---

### 🖼️ Handling Different Image Modes (Pillow)

When loading images with **Pillow**, the image *mode* specifies how pixel values are represented:

| Mode | Meaning | Description |
|------|---------|-------------|
| **`"L"`** | 8-bit grayscale | One channel with values in **[0–255]**. |
| **`"I"`** | 32-bit grayscale (integer) | One channel, often used in scientific TIFFs; can store large dynamic ranges. |
| **`"I;16"`** | 16-bit grayscale (unsigned integer) | Common in microscopy and CT data; pixel values in **[0–65535]**. |
| **`"RGB"`** | Red–Green–Blue | Standard 3-channel color image. |
| **`"RGBA"`** | RGB + Alpha | 4 channels, includes transparency. |

To ensure compatibility with SAM2, images are always converted to **RGB**:

- `"L"`, `"I"`, or `"I;16"` → converted to RGB by duplicating the grayscale channel  
- `"RGBA"` → alpha channel removed and converted to RGB  

This ensures the consistent format **`W × H × 3`** required by SAM2.

---

### 🧮 Accepted Bit Depths

SAM2 accepts:

- **8-bit images** (0–255)
- **16-bit images** (0–65535)
- **32-bit integer** grayscale images

During preprocessing, SAM2 automatically:

- converts arrays to **float32**
- normalizes values to **[0, 1]**  
- downscales images larger than **1024 × 1024** (aspect ratio preserved)
- upsamples segmentation masks back to the original resolution

---

### 📁 Loading Your Own Image

You can upload an image using the file navigator on the left:

- **Google Colab:** uploaded images appear under `/content/images/`
- **Local environment:** the working directory is the folder containing this notebook

Please update the `file_path` variable accordingly 🚀


In [ ]:
from google.colab import files

uploaded = files.upload()

if uploaded:
    file_path = list(uploaded.keys())[0]
else:
    print("No file uploaded, using example image")
    # File path
    file_path = "/content/drive/MyDrive/PP15_15_100x copy3.jpg"



In [ ]:


# Load image
image = Image.open(file_path)

# Normalize to RGB
# Handle grayscale modes: "L" (8-bit), "I" (32-bit), "I;16" (16-bit unsigned)
if image.mode in ("L", "I", "I;16"):
    image = image.convert("RGB")

# Handle RGBA (drop alpha)
elif image.mode == "RGBA":
    image = image.convert("RGB")

# Convert to NumPy
image_np = np.asarray(image, copy=True)

# Print info (correct axis order: H x W x C)
h, w, c = image_np.shape
print(f"Image shape: H {h}, W {w}, C {c}, dtype={image_np.dtype}")

In [ ]:
## Show the Image
plt.figure(figsize=(4, 4))
plt.imshow(image_np)
plt.axis('off')
plt.show()

**Optional**: Center crop the image to match the input size

In [ ]:
## center crop the image
def center_crop_np(image: np.ndarray, crop_size=512):
    """Fast center crop for a NumPy image.

    Args:
        image (np.ndarray): Input image as a NumPy array (H, W) or (H, W, C).
        crop_size (int): Target crop size (assumes square crop).

    Returns:
        np.ndarray: Center-cropped image.
    """
    h, w = image.shape[:2]  # Get height and width
    ch, cw = crop_size, crop_size  # Crop size

    # Compute center crop coordinates
    start_h = max((h - ch) // 2, 0)
    start_w = max((w - cw) // 2, 0)
    end_h = min(start_h + ch, h)
    end_w = min(start_w + cw, w)

    return image[start_h:end_h, start_w:end_w]


image_np = center_crop_np(image_np)
## Show the Image
plt.figure(figsize=(4, 4))
plt.imshow(image_np)
plt.axis('off')
plt.show()

## ⚙️ Automatic mask generation
To run fully automatic segmentation, we first load a SAM 2 model checkpoint and then wrap it with the `SAM2AutomaticMaskGenerator`.  
This generator densely samples prompts over the image (e.g. a grid of points) and lets the model propose masks without any manual interaction.

### Steps

1. **Select a SAM 2 variant**  
   - The `model_cfg` points to a YAML config describing the architecture (e.g. `sam2.1_hiera_t`).
   - The `sam2_checkpoint` is the corresponding `.pt` weight file.  
   Make sure both paths match the same model variant.

2. **Build the SAM 2 model**  
   `build_sam2(model_cfg, sam2_checkpoint, device=device)` loads the model on the selected device (CPU, CUDA, or MPS).

3. **Configure the automatic mask generator**  
   The `SAM2AutomaticMaskGenerator` controls:
   - how densely prompts are sampled across the image (`points_per_side`),
   - GPU/memory usage (`points_per_batch`),
   - which masks are kept or filtered (`pred_iou_thresh`, `min_mask_region_area`, `box_nms_thresh`, `stability_score_thresh`).

These parameters let you trade off **speed vs. mask density** and **aggressiveness of filtering**.  
For micrographs with many fine structures (e.g. fibers, pores, defects), slightly higher `points_per_side` and lower thresholds often (`stability_score_thresh`, `pred_iou_thresh`, or `box_nms_thresh`) help capture more detailed masks.

In [ ]:
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator
from sam2.addons import show_anns

# set here your checkpoint path and path to yourmodel cfg
sam2_checkpoint = "./checkpoints/sam2.1_hiera_tiny.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_t.yaml"

sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=False)

mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,                 # The preloaded SAM2 model used for automatic segmentation

    points_per_side=78,         # Sampling density of prompt points per image side.
                                # A higher value → more points → denser coverage, better for small or fine structures,
                                # but also slower and more memory-intensive.

    points_per_batch=64,        # Number of sampled points processed simultaneously by the model.
                                # Controls batch size; balance between speed and GPU memory usage.

    pred_iou_thresh=0.1,        # Minimum predicted IoU between the mask and its region proposal
                                # required to keep a candidate. Lower this to retain weaker (small or uncertain) masks.

    min_mask_region_area=40,    # Minimum area (in pixels) for a mask to be kept.
                                # Increasing this removes tiny specks; decreasing it helps keep very small objects.

    box_nms_thresh=0.3,         # Non-Maximum Suppression (NMS) threshold for filtering overlapping boxes.
                                # Lower values make the filter stricter, keeping fewer overlapping regions.

    stability_score_thresh=0.9  # Confidence threshold measuring mask consistency under perturbations.
                                # Lowering this (e.g. 0.7–0.8) retains more masks, especially smaller or noisier ones.
)

### ⚙️ Detection - Results
To generate masks, call the generate() function on an input image.

The returned object masks is a list of dictionaries, where each dictionary describes one detected region (e.g., a fiber cross-section). Each entry contains both the segmentation mask and associated metadata:

- `segmentation` : binary mask of the segmented region
- `area` : pixel area of the mask
- `bbox` : bounding box coordinates in the format `[x, y, width, height]`
- `predicted_iou` : model’s confidence estimate of segmentation accuracy (Intersection-over-Union (Jaccard) score)
- `point_coords` : coordinates of the prompt points used (if any)
- `stability_score` : measure of segmentation consistency under perturbations
- `crop_box` : image region from which the mask was generated

This output provides both the segmentation itself and useful metadata for filtering, analysis, or visualization.

In [ ]:
# Detect for one image
print(f"Detection started...")
t0 = time()
masks = mask_generator.generate(image_np)
t = time() - t0
print(f"Detection finished in {t:.3f} seconds...")
print(f"Detected objects: {len(masks)}")

### 🛠️ Post-processing: Removing Spurious Background Masks

In some cases, SAM2 may also generate a mask that corresponds to the image background.
This typically appears as a very large mask that covers most of the image but is not a meaningful object (e.g., not a fiber cross-section).

To filter these out, we sort all detections by their area and remove the largest one if it exceeds a chosen threshold.

In [ ]:
# Sort masks by area (largest first)
sorted_masks = sorted(masks, key=lambda m: m["area"], reverse=True)

# Remove the largest mask if it is likely background (area > 1e5 px)
if sorted_masks and sorted_masks[0]["area"] > 1e5:
    sorted_masks = sorted_masks[1:]

### 🖼️ Vizualization

In this step, we display the detected instances on top of the original image.  
Each mask is assigned a **random color** sampled from the selected Matplotlib colormap (`plasma` in this example).  
You can freely change the colormap to any Matplotlib-supported option (e.g., `viridis`, `magma`, `inferno`, `tab20`, …).  

The transparency of the overlaid masks is controlled by the `alpha` parameter.  
Increasing `alpha` makes the masks more opaque; decreasing it makes the underlying image more visible.

In [ ]:
# Display segmented masks on top of the image
plt.figure(figsize=(5, 5))
plt.imshow(image_np)

# Random-color instance masks using the "plasma" colormap
label_canvas = show_anns(
    sorted_masks,
    alpha=0.5,          # overlay transparency (0=transparent, 1=opaque)
    cmap_key="plasma"   # Matplotlib colormap for mask colors
)

plt.axis('off')
plt.show()

## 📊 Statistics

Using the `label_map`, we can now extract quantitative information about the segmented fiber cross sections.
For this purpose, we apply the function `regionprops` (`regionprops_table` returns a pandas dataframe) from the `scikit-image` package, which computes a wide range of morphological and intensity-based statistics for each labeled region (comparable to 1D-descriptors).

Typical properties include:
- Area: number of pixels belonging to each fiber
- Centroid: spatial position of the fiber center
- Equivalent diameter: diameter of a circle with the same area as the fiber
- Eccentricity: measure of how elongated the fiber cross section is
- Orientation: local rotation of the fitted ellipse

These descriptors enable detailed quantitative analysis of fiber size, shape, and spatial distribution within the micrograph.

### 🛠️ Pre-Processing

After generating instance masks with SAM2, each detected object is represented as a separate entry in the `masks` list.  
Each entry contains a boolean `segmentation` array indicating which pixels belong to that specific region.

For further analysis—such as computing descriptors, region properties, training segmentation models, or exporting results—we convert these masks into a **single label map**, where:

- each pixel is assigned an **integer class ID**
- each instance gets a **unique identifier** (1, 2, 3, …)
- background pixels remain **0**

This label map is the standard representation used by many image-processing libraries (e.g., scikit-image), machine learning pipelines, and downstream workflows like meshing or statistical analysis.

In [ ]:
def masks_to_labelmap(masks, image_shape):
    """
    Convert a list of SAM2 mask dictionaries to a label map.

    Parameters
    ----------
    masks : list of dict
        Each dictionary must contain a boolean array under the key 'segmentation'.
        Each mask corresponds to one detected instance.

    image_shape : tuple
        Target label map shape (height, width).

    Returns
    -------
    label_map : np.ndarray, dtype=int16
        A labeled image where:
        - 0 = background
        - 1..N = unique instance IDs for each mask in `masks`
    """
    label_map = np.zeros(image_shape, dtype=np.int16)

    for i, mask_dict in enumerate(masks, start=1):
        mask = mask_dict["segmentation"]
        label_map[mask] = i

    return label_map

In [ ]:
# Extract spatial dimensions (height, width)
image_shape = image_np.shape[:2]

# Convert SAM2 instance masks into a single label map
label_map = masks_to_labelmap(sorted_masks, image_shape)

# Visualize the label map
plt.figure(figsize=(4, 4))
plt.imshow(label_map, cmap="plasma")  # each instance visualized with a unique color
plt.title("Instance Label Map")
plt.axis("off")
plt.show()

### 🔬 Region Property Extraction

Once the instance label map is generated, we can compute geometric and shape descriptors for each segmented region.  
Using `skimage.measure.regionprops_table`, we extract a set of quantitative properties such as:

- **Area**, **perimeter**, **centroid**  
- **Bounding box dimensions**  
- **Major/minor axis lengths** (ellipse approximation)  
- **Eccentricity** (shape elongation)  
- **Convex area** and **solidity** (compactness)  
- **Equivalent diameter**  
- **Maximum Feret diameter** (max caliper distance)  
- **Hu moments** (rotation-invariant shape descriptors)

These features are extremely useful for fiber characterization, quality inspection, and downstream analysis such as clustering, filtering, or statistical modeling.  
The results are stored in a Pandas DataFrame to make them easy to inspect, visualize, and summarize.

In [ ]:
# Compute geometric and morphological properties of each labeled region
result_dict = regionprops_table(
    label_map,
    properties=(
        "label",
        "area",
        "bbox",
        "centroid",
        "convex_area",
        "eccentricity",
        "equivalent_diameter",
        "extent",
        "feret_diameter_max",
        "filled_area",
        "major_axis_length",
        "minor_axis_length",
        "orientation",
        "perimeter",
        "solidity",
        "moments_hu",
    ),
)

# Convert the dictionary of arrays to a Pandas DataFrame
props_df = pd.DataFrame(result_dict)

# Summary of detected regions (e.g., fibers)
print(f"Extracted {len(props_df)} fiber regions.")

# Display first rows for inspection
props_df.head()

### 💾 Save the Results to CSV/XLSX

After extracting geometric and morphological properties for each detected fiber region, it is often useful to export the results for further analysis, documentation, or reporting.

Saving the table to **CSV** allows quick inspection in text editors or Python/R scripts, while **Excel (XLSX)** is convenient for sharing results with colleagues or performing additional analysis in spreadsheet software.

Below, we save the `props_df` DataFrame to both formats.  
You can adjust the file paths as needed.

In [ ]:
# Define output file paths
csv_path = "./fiber_properties.csv"
xlsx_path = "./fiber_properties.xlsx"

# Save as CSV
props_df.to_csv(csv_path, index=False)
print(f"Saved CSV to: {csv_path}")

# Save as Excel
props_df.to_excel(xlsx_path, index=False)
print(f"Saved Excel file to: {xlsx_path}")

### 🔬 Fiber Size Distribution Analysis

The **equivalent diameter** is a useful scalar descriptor that represents the diameter of a circle having the same area as the segmented region.  
For fiber cross-sections, this provides a convenient approximation of fiber size, even if the actual shape is slightly elliptical.

In this step:

- we extract the *equivalent diameter* for all regions with sufficient area  
  (here we filter out very small detections with `area > 3000` to remove noise or artifacts)
- we visualize the size distribution using a histogram

This helps assess fiber uniformity, detect anomalies, or compare batches of material.

In [ ]:
# Extract equivalent diameters, filtering out small/noisy regions
equiv_diameters = props_df["equivalent_diameter"][props_df["area"] > 3000]

# Plot histogram of fiber equivalent diameters
plt.figure(figsize=(6, 4))
plt.hist(
    equiv_diameters,
    bins=30,
    facecolor="purple",
    edgecolor="black",
    alpha=0.7,
)

plt.title("Distribution of Fiber Equivalent Diameters")
plt.xlabel("Equivalent Diameter [px]")
plt.ylabel("Count")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 👩‍🔬 Higher-Order Descriptor ($S_2$)

The two-point correlation function $S_2$ is a higher-order statistical descriptor that captures **spatial relationships** within a microstructure.  
Unlike simple metrics such as volume fraction, $S_2(r)$ quantifies how phase information (e.g., fiber vs. matrix) **correlates as a function of distance**.  
This allows us to measure clustering, characteristic length scales, anisotropy, periodicity, and the overall spatial order of the material.

---

### FFT-Based Computation

In practice, $S_2$ is evaluated using an **FFT-based convolution**, which enables fast and accurate correlation estimation even for large images.  
For a binary phase indicator function $I(\mathbf{x})$, the descriptor is computed as:

$$
S_2(\mathbf{r}) = \mathcal{F}^{-1} \left( \mathcal{F}(I)\,\cdot\,\mathcal{F}(I)^{*} \right)
$$

where:

- $\mathcal{F}$ is the Fast Fourier Transform  
- $\mathcal{F}(I)^{*}$ is its complex conjugate  

This formulation ensures efficient correlation computation with minimal numerical overhead.

---

### Meaning of the Variables

- **Binary map**  
  $ I(\mathbf{x}) \in \{0,1\} $: indicator function for the phase of interest (1 = fiber, 0 = background).

- **$S_2(x, y)$ — 2D correlation map**  
  Probability that two points separated by vector $(x, y)$ belong to the same phase.

- **$S_2(r)$ — radial correlation**  
  Radial average of the 2D map, producing a 1D profile showing how similarity decays with distance.

---

### What $S_2$ Reveals

- Clustering intensity and characteristic scale  
- Correlation length of the microstructure  
- Presence of periodic or directional patterns  
- Typical microstructural feature size  
- Degree of randomness vs. spatial order  

Because of its FFT-based formulation, $S_2$ is widely used in materials science for **microstructure characterization**, **RVE determination**, and **structure–property modeling**.

---

### Computing the Two-Point Correlation Function in This Workflow

1. Convert the instance label map into a **binary phase map**  
   (1 = fiber, 0 = background).

2. Use `s2_descriptor()` to compute:  
   - **`S2`** — full 2D FFT-based correlation  
   - **`S2r`** — radial average $ S_2(r) $

3. Convert tensors to NumPy arrays for visualization and further analysis.

These descriptors help quantify spatial patterns, detect clustering, and compare microstructural samples based on their statistical structure.

In [ ]:
from sam2.addons import s2_descriptor

# Convert the instance label map to a binary phase map
# (1 = foreground/fibers, 0 = background)
binary_map_tensor = torch.tensor((label_map > 0), dtype=torch.long).unsqueeze(0)

# Compute the full 2D S2 descriptor and its radial average
S2, S2r = s2_descriptor(binary_map_tensor, radial=True)

# Convert tensors to NumPy arrays for plotting or further processing
binary_np = binary_map_tensor.squeeze(0).cpu().numpy()
s2_np     = S2.squeeze(0).cpu().numpy()
s2r_np    = S2r.squeeze().cpu().numpy()

#### 🖼️ Visualization

In [ ]:
# Plot
fig, axs = plt.subplots(1, 3, figsize=(14.4, 4.8))

# --- Binary map ---
axs[0].imshow(binary_np, cmap="plasma")
axs[0].set_title("Binary Mask")
axs[0].set_xlabel(r"x")
axs[0].set_ylabel(r"y")

# --- S2 2D map ---
im1 = axs[1].imshow(s2_np, cmap="plasma")
axs[1].set_title("S₂ Descriptor (2D)")
axs[1].set_xlabel(r"x")
axs[1].set_ylabel(r"y")
# horizontal colorbar below the image
cbar1 = fig.colorbar(
    im1,
    ax=axs[1],
    orientation="horizontal",
    fraction=0.046,   # size of the bar relative to the axis
    pad=0.08          # spacing between image and colorbar
)
cbar1.set_label("S₂ Value", labelpad=6)

# --- Radial S2 ---
axs[2].plot(s2r_np[1:], color="tab:purple", label="Radial $S_2(r)$")
axs[2].scatter(1, s2r_np[1], marker="s", color="tab:purple", label=r"$S_2(r=1)=\phi$")

axs[2].set_xlim(-50, 500)
axs[2].set_ylim(0.4, 0.8)
axs[2].set_title("Radial S₂")
axs[2].set_xlabel("r (pixels)")
axs[2].set_ylabel("S₂(r)")
axs[2].grid(True)
axs[2].legend(loc="best")

plt.tight_layout()
plt.show()

### 🧪 RVE Estimation from $S_2(r)$

After computing the two-point correlation function $S_2$, we can derive several important **statistical descriptors** that provide insight into the microstructure and help determine the appropriate **Representative Volume Element (RVE)** size.

This section computes:

---

#### **1. Volume Fraction $ \phi $**  
The simplest descriptor—fraction of pixels belonging to the fiber phase.  
Used as a baseline for correcting $S_2$ and for RVE scaling laws.

---

#### **2. Correlation Length (Half-Height Method)**  
Using the radial two-point correlation $S_2(r)$, the correlation length is defined as the distance at which $S_2(r)$ has decayed to:

$$
\frac{ \phi + \phi^2 }{2}
$$

This length scale characterizes:
- the typical size of microstructural features  
- heterogeneity and clustering  
- spatial coherence in the material  

---

#### **3. Integral Range $A_{\text{int}}$**  
The **integral range** is the area under the corrected correlation function and quantifies how spatially extended correlations are:

$$
A_{\text{int}} = \int_0^\infty \left[ S_2(r) - \phi^2 \right] \, 2\pi r \, dr
$$

It represents the statistical "patch size" of the microstructure and is essential for estimating the RVE.

---

#### **4. RVE Size Based on Target Statistical Accuracy (COV)**  
Using the integral range, volume fraction, and a desired **coefficient of variation (COV)** (e.g., 5% or 10%), we estimate the required RVE side length:

$$
L_{\text{RVE}} \approx \sqrt{ \frac{ \phi (1-\phi) A_{\text{int}} }{ \mathrm{COV}^2 } }
$$

This tells us how large a microstructural window must be to achieve a statistically reliable estimate of homogenized properties.

Common targets:
- **COV = 0.05 → 5% scatter**  
- **COV = 0.10 → 10% scatter**

---

Together, these descriptors provide a quantitative link between microstructure and statistical representativeness, and they guide the selection of RVEs for simulations or material modeling.

In [ ]:
from sam2.addons import (
    s2_descriptor,
    phi_descriptor,
    corr_length_halfheight,
    integral_range_from_S2r,
    rve_size_from_integral_range,
)

# --- 1) Binary phase map (1 = fibers, 0 = background) ---
binary_map_tensor = torch.tensor(label_map > 0, dtype=torch.long).unsqueeze(0)

# --- 2) Statistical descriptors: S2, S2(r), volume fraction, correlation length ---
s2, s2r = s2_descriptor(binary_map_tensor, radial=True)
phi     = phi_descriptor(binary_map_tensor)
corrlen = corr_length_halfheight(s2r, phi)

# Convert to NumPy for plotting or external analysis
binary_np = binary_map_tensor.squeeze(0).cpu().numpy()
s2_np     = s2.squeeze(0).cpu().numpy()
s2r_np    = s2r.squeeze().cpu().numpy()

# --- 3) Integral range A_int (in px²) ---
A_int = integral_range_from_S2r(s2r, phi)

# --- 4) Target coefficient of variation (COV) ---
cv_target = 0.05  # e.g. 5% scatter (use 0.10 for 10%)

# --- 5) RVE side length in pixels ---
L_rve_px = rve_size_from_integral_range(phi, A_int, cv_target)

# --- 6) Print summary ---
print(f"phi_mean           = {phi.item():.4f}")
print(f"correlation length = {corrlen.item():.4f} px")
print(f"A_int_mean         = {A_int.item():.2f} px²")
print(f"Target COV         = {cv_target:.3f}")
print(f"RVE side           = {L_rve_px.item():.1f} px")

In [ ]:
# Plot
fig, axs = plt.subplots(1, 3, figsize=(14.4, 4.8))

# --- Binary map ---
axs[0].imshow(binary_np, cmap="plasma")
axs[0].set_title("Binary Mask")
axs[0].set_xlabel(r"x")
axs[0].set_ylabel(r"y")

# --- S2 2D map ---
im1 = axs[1].imshow(s2_np, cmap="plasma")
axs[1].set_title("S₂ Descriptor (2D)")
axs[1].set_xlabel(r"x")
axs[1].set_ylabel(r"y")
# horizontal colorbar below the image
cbar1 = fig.colorbar(
    im1,
    ax=axs[1],
    orientation="horizontal",
    fraction=0.046,   # size of the bar relative to the axis
    pad=0.08          # spacing between image and colorbar
)
cbar1.set_label("S₂ Value", labelpad=6)

# --- Radial S2 ---
axs[2].plot(s2r_np[1:], color="tab:purple", label="Radial $S_2(r)$")
axs[2].scatter(1, s2r_np[1], marker="s", color="tab:purple", label=r"$S_2(r=1)=\phi$")

axs[2].axvline(corrlen, ls="--",label=r"$corr/2$")  # mark correlation length
axs[2].axvline(L_rve_px.item(), ls="-",label=r"$L_{\text{RVE}}$")  # mark correlation length
axs[2].axhline(phi.item()**2, ls="--",color="tab:orange",label=r"$\phi^2$")  # mark correlation length


axs[2].set_xlim(-50, 500)
axs[2].set_ylim(0.4, 0.8)
axs[2].set_title("Radial S₂")
axs[2].set_xlabel("r (pixels)")
axs[2].set_ylabel("S₂(r)")
axs[2].grid(True)
axs[2].legend(loc="upper right")

plt.tight_layout()
plt.show()

# 2️⃣ Part 2: Panorama Analysis

## 💿 Load Panorama
For large micrographs—such as stitched optical images, panoramic scans, or high-resolution CT slices—the full image often exceeds the memory limits of a single forward pass through SAM2.  
To address this, we load the complete micrograph and later subdivide it into **overlapping patches** of size **1024 × 1024 pixels** with a specific **stride**.

This patch-based approach provides several advantages:

- **Memory efficiency:** Large images are processed in manageable tiles.
- **Improved local accuracy:** Each patch is processed at full resolution, preserving fine details.
- **Better segmentation of heterogeneous regions:** Local variations in fiber density, texture, or illumination are captured more effectively.
- **Enables spatially resolved analysis:** We can compute descriptors (fiber size, orientation, cluster structure, \(S_2\), φ, etc.) **per patch**, allowing microstructure mapping across the sample.

Below, we load the panorama image, normalize it to RGB, and convert it to a NumPy array for further patch extraction.

In [ ]:
file_path_pano = "./images/panorama.tif"

pano = Image.open(file_path_pano)

if pano.mode in ("L", "I"):  # grayscale
    pano = pano.convert("RGB")
elif pano.mode == "RGBA":    # drop alpha
    pano = pano.convert("RGB")

pano_np = np.array(pano)
print(f"Image has the shape {pano_np.shape} and datatype {pano_np.dtype}")

plt.figure(figsize=(10,4))
plt.imshow(pano_np)
plt.axis("off")
plt.show()

✂️ Optional Demo Cropping

To keep the example lightweight, we optionally crop the large panorama for demo purposes 🧪:
- Original panorama size: 2636 × 16140 px
- Full-resolution inference takes ~13 seconds per image ⚠️
- With a 1024 px patch size and a conservative stride, we’d still need:
$$\left\lceil \frac{16140}{1024} \right\rceil = 16 \text{ patches along the width}\quad \approx 5 \textrm{min}$$
- Cropping keeps it computationally manageable while preserving the core logic.

In [ ]:
# centercrop to a smaller panorama shape for demo
# we use here 2x the maximum image size

pano_np = center_crop_np(pano_np, crop_size=2*1024)
print(f"new shape of the panorama {pano_np.shape}")
## Show the Image
plt.figure(figsize=(10, 4))
plt.imshow(pano_np)
plt.axis('off')
plt.show()

## 🧩 Patch the Image

Large panoramic micrographs cannot be processed in a single pass without exceeding GPU memory.  
To handle this, we **tile (patch)** the input image into overlapping windows and later **stitch** the per-patch SAM2 detections back into a global instance map.

The `Sam2Patcher` helper takes care of:

- **Tiling** the image into patches of size `patch_h × patch_w`
- Using **overlaps** (`overlap_h`, `overlap_w`) to avoid artifacts at patch borders.  
  This creates an effective **stride** of  
  $$
  \text{stride\_h} = \text{patch\_h} - \text{overlap\_h}, \quad
  \text{stride\_w} = \text{patch\_w} - \text{overlap\_w},
  $$
  ensuring that each region of the image is seen multiple times.  
  Overlaps help SAM2 maintain segmentation quality near patch boundaries and prevent discontinuities when stitching.

- Optionally **padding** images that are not divisible by the patch size, ensuring the entire image is processed without distortion.

- **Merging instances across tiles** based on their Intersection-over-Union (IoU).  
  This step is crucial because the same fiber may appear in two or more overlapping patches; the `merge_iou` threshold determines when two detections should be considered the same global instance.

Together, this tiling–stitching workflow enables high-resolution, memory-efficient segmentation of very large micrographs (including panoramas, stitched mosaics, and large CT slices) while preserving consistent instance IDs across the full field of view.

This enables scalable, high-resolution segmentation of large micrographs while keeping GPU memory usage under control and maintaining

In [ ]:
from sam2.addons import Sam2Patcher

# Initialize tiler for 1024×1024 patches with 6.25% overlap
patcher = Sam2Patcher(
    patch_h=1024,
    patch_w=1024,
    overlap_h=64,
    overlap_w=64,
    pad_mode="constant",  # how to pad if image size is not divisible by patch size
    merge_iou=0.1         # IoU threshold to merge instances across neighboring patches
)

The tiling step returns three components:

1. **`patches`**  
   A list of image tiles, each typically of size `1024 × 1024 × 3` (unless trimmed at boundaries).  
   Each patch can be processed independently by SAM2, making it possible to segment very large images that wouldn't fit into GPU memory at once.

2. **`offsets`**  
   A list of `(y, x)` coordinates, one for each patch.  
   These indicate the **top-left position of each patch in the original panorama**, allowing us to later reconstruct a full-resolution global instance map by placing each segmented patch back into its original location.

3. **`padded_shape`**  
   The shape `(H, W, 3)` of the full padded image that the patches cover.  
   Padding is introduced only when the image dimensions are not exact multiples of the patch size.  
   This ensures full coverage without distortion or cropping.

Together, these outputs form the basis for a **tile → segment → stitch** pipeline:
- SAM2 is run on each patch independently.
- Results are merged using the offsets.
- The final stitched instance map reconstructs the entire panoramic micrograph with consistent IDs.

In [ ]:
## Patching
patches, offsets, padded_shape = patcher.tile_numpy(pano_np)
print(f"Tiled into {len(patches)} patches of shape {patches[0].shape}, \
      \npadded shape: {padded_shape}")

### 🖼️ Display some random Patches

In [ ]:
# Number of patches to display (max 4)
num_show = min(4, len(patches))

# Create figure
fig, axs = plt.subplots(
    1, num_show,
    figsize=(num_show * 4.8, 4.8)
)
axs = np.array(axs).ravel()  # ensure iterable

# Random visualization of patches and detections
for ax in axs:
    idx = np.random.randint(0, len(patches))

    ax.imshow(patches[idx])
    ax.set_title(f"Patch {idx:03d}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## ⚙️ Automatic Mask Generation per Patch

Loop over all patches and run SAM2 to generate segmentation masks for the detected objects.
For each patch, SAM2 produces a list of detections—one dictionary per instance—containing the binary mask, bounding box, area, and additional metadata.

All patch-wise results are stored in a **list of lists**:

- the outer list corresponds to the sequence of patches  
- each inner list contains the instance dictionaries detected within that patch  

This structured collection allows us to later **stitch** all local detections back into a single global instance map, ensuring that overlapping or duplicated detections across patch boundaries are correctly merged.

In [ ]:
#----- Mask Generation -----
all_detections = []
for i, p in enumerate(tqdm(patches,
    total=len(patches),
    desc="Mask generation",
    unit="patch"
)):
    ## detect masks per patch
    detections = mask_generator.generate(p)
    print(f"detected {len(detections)} for patch {i:03d}")
    ## gather results
    detections = sorted(detections, key=lambda x: x["area"],reverse=True)

    all_detections.append(detections[1:])

### 🖼️ Visualize Sample Patch Segmentations

Below we display a few (max 4) randomly selected patches along with their corresponding SAM2 segmentation masks.  

In [ ]:
# Number of patches to display (max 4)
num_show = min(4, len(patches))

# Random visualization of patches and detections
for i in range(num_show):
    idx = np.random.randint(0, len(patches))
    plt.figure()
    plt.imshow(patches[idx])
    show_anns(all_detections[idx])
    plt.title(f"Patch {idx:03d}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## 🪡 Stitching the Results

Once all patches have been processed individually, their local SAM2 detections must be merged back into a **single global instance map**.  
The `Sam2Patcher` handles this by aligning each patch’s detections using its offset and resolving overlaps using an IoU-based merging rule.

The output of this step consists of:

- **`label_map`**  
  A full-resolution 2D array where each pixel contains the **global instance ID** (0 = background).

- **`instances`**  
  A list of dictionaries, one per detected object in the entire panorama.  
  Each entry contains:
  - `"id"` — the unique global instance ID  
  - `"area"` — number of pixels belonging to the instance  
  - `"segmentation_crop"` — the boolean mask cropped to its bounding box for compact storage  
  - `"bbox"` — bounding box in `[x0, y0, width, height]` format (global coordinates)

This stitched output provides a unified and consistent segmentation map across the whole image, even when individual patches overlap or contain partial views of the same object.

In [ ]:
# Stitch per-patch SAM2 detections into a global label map and instance list
label_map, instances = patcher.stitch_sam2_instances(
    padded_shape_hw=padded_shape[:2],  # (H, W) of the full (padded) canvas
    offsets=offsets,                   # top-left (y, x) coords of each patch
    per_tile_sam2=all_detections,      # list of SAM2 detections per patch
    debug=False,
)

### 🖼️ Display the Results for Panorama

In [ ]:
plt.figure(figsize=(4,4))
plt.imshow(pano_np)
show_anns(instances)
plt.axis("off")
plt.show()

## 💾 Save the Results

You can export the final `label_map` as a `*.tiff` file for use in tools like **Fiji/ImageJ**.  
These tools treat each integer value as a separate label, allowing easy visualization and verification of the full-resolution segmentation.

Once loaded in Fiji, the label map can be:
- 🎨 shown with a color LUT (e.g., Glasbey)  
- 👁️ overlaid onto the original micrograph  
- 🔍 inspected region by region  
- 📏 used for measurements or plugin workflows  
- ✏️ edited manually where needed  

For more advanced refinement, **LabelKit** offers tools to:
- 🔁 merge or split instances  
- ✂️ correct boundaries  
- 🧼 remove false positives  
- ➕ add missing segments  
- 💾 export cleaned annotations (NumPy, PNG/TIFF, JSON)

These refined masks are perfect for creating **high-quality training data** to **fine-tune SAM2** on your specific microstructure — leading to more accurate and robust segmentations on challenging samples.

In [ ]:
Image.fromarray(label_map).save("results.tiff")

## 💾🚀🧪 Save the Results *.hdf5

For **very large images** or full **panoramas**, it is recommended to save the results as an `*.hdf5` container instead of TIFF.  
HDF5 offers several advantages:

- 📦 efficient storage for large 2D/3D arrays  
- ⚡ fast, chunked read/write access  
- 🧩 ability to store multiple datasets (e.g., `label_map`, `instances`, `original_image`)  
- 📝 support for metadata such as pixel size, ROI info, or processing parameters  
- 🔒 robust format for long-term archival and reproducibility  

This makes HDF5 ideal when working with:
- gigapixel micrographs  
- stitched panoramic scans  
- large CT slices or full 3D volumes  
- workflows that involve patch-based segmentation and descriptor computation  

Saving in HDF5 ensures your data remains organized, scalable, and easy to load back into Python for downstream analysis such as descriptor extraction, correlation functions, or RVE estimation.

In [ ]:
from sam2.addons import write_h5
dict_out = {
    "image": image_np ,
    "labels": label_map,
    "binary": np.array(label_map > 0 ,dtype=np.uint8), ## optional saving the binary label map
    "instances": instances
}

write_h5(path = "results.h5", dict_out= dict_out)